# Repository Indexing Crew

This notebook demonstrates the multi-agent CrewAI system for analyzing, annotating, and indexing code repositories into vector stores.

## Overview

The Repository Indexing Crew consists of four specialized agents:

1. **Code Analyzer Agent** - Analyzes repository structure, languages, and frameworks
2. **Documentation Generator Agent** - Creates documentation for code
3. **Code Annotator Agent** - Adds semantic annotations and metadata
4. **Orchestrator Agent** - Coordinates the workflow and handles vector storage

## Features

- Multi-agent collaboration using CrewAI
- Dual vector store support (LanceDB + ChromaDB)
- MLFlow experiment tracking
- OpenTelemetry distributed tracing
- Parameterized embeddings (Ollama, Sentence Transformers, OpenAI)


## Setup


In [ ]:
# Imports
from pathlib import Path

from agentic_assistants import AgenticConfig, OllamaManager
from agentic_assistants.crews import RepositoryIndexingCrew
from agentic_assistants.crews.telemetry import get_crew_telemetry
from agentic_assistants.embeddings import EmbeddingProvider
from agentic_assistants.patterns import RAGPattern
from agentic_assistants.utils.logging import setup_logging

# Setup
setup_logging(level="INFO")
config = AgenticConfig()

# Ensure Ollama is running
ollama = OllamaManager(config)
ollama.ensure_running()
model = ollama.ensure_model()

print(f"✓ Configuration loaded")
print(f"✓ Using model: {model}")
print(f"✓ Vector backend: {config.vectordb.backend}")
print(f"✓ MLFlow enabled: {config.mlflow_enabled}")


## 1. Create the Indexing Crew


In [ ]:
# Create the repository indexing crew
crew = RepositoryIndexingCrew(
    config=config,
    model=model,
    vector_backend="lancedb",  # or "chroma" or "both"
    embedding_model="nomic-embed-text",
    verbose=True,
)

print(f"✓ Crew created")
print(f"  Model: {crew.model}")
print(f"  Backend: {crew.vector_backend}")
print(f"  Embedding: {crew.embedding_model}")


## 2. Select Repository to Index

Specify the path to a local repository you want to index.


In [ ]:
# Set the repository path
# Change this to your target repository
REPO_PATH = "../src"  # Index this project's source code

# Resolve and validate
repo_path = Path(REPO_PATH).resolve()
print(f"Repository: {repo_path}")
print(f"Exists: {repo_path.exists()}")

if repo_path.exists():
    # Count files
    py_files = list(repo_path.glob("**/*.py"))
    print(f"Python files found: {len(py_files)}")


## 3. Run the Indexing Crew

This will:
1. Analyze the repository structure
2. Generate documentation
3. Create semantic annotations
4. Index everything to the vector store

**Note:** This process may take several minutes depending on repository size.


In [ ]:
# Run the indexing crew
# This creates an MLFlow experiment run

COLLECTION = "agentic-source"  # Collection name in vector store
EXPERIMENT = "repo-indexing-notebook"

print("Starting repository indexing...")
print(f"Collection: {COLLECTION}")
print(f"Experiment: {EXPERIMENT}")
print("="*50)

result = crew.run(
    repo_path=repo_path,
    collection=COLLECTION,
    experiment_name=EXPERIMENT,
    tags={"source": "notebook", "version": "1.0"},
)

print("="*50)
print(f"Success: {result.success}")
print(f"Duration: {result.duration_seconds:.2f}s")


## 4. View Results


In [ ]:
# View indexing results
print("Indexing Results")
print("="*50)

print(f"\nStatus: {'✓ Success' if result.success else '✗ Failed'}")
print(f"Repository: {result.repo_path}")
print(f"Collection: {result.collection}")
print(f"Duration: {result.duration_seconds:.2f} seconds")

if result.stats:
    print("\nStatistics:")
    for key, value in result.stats.items():
        print(f"  {key}: {value}")

if result.errors:
    print("\nErrors:")
    for error in result.errors:
        print(f"  ⚠ {error}")

if result.experiment_run_id:
    print(f"\nMLFlow Run: {result.experiment_run_id}")
    print(f"View at: {config.mlflow.tracking_uri}")


## 5. Search the Indexed Repository

Now you can search the indexed codebase using semantic search.


In [ ]:
# Search the indexed repository
query = "How does the vector store work?"

print(f"Query: {query}")
print("="*50)

results = crew.search(
    query=query,
    collection=COLLECTION,
    top_k=5,
)

for i, r in enumerate(results, 1):
    print(f"\n[{i}] Score: {r.score:.3f}")
    file_path = r.document.metadata.get("file_path", "unknown")
    print(f"    File: {file_path}")
    print(f"    Content: {r.document.content[:200]}...")


## 6. RAG Pattern - Ask Questions About the Code

Use the RAG pattern to ask questions about the indexed codebase.


In [ ]:
# Create RAG pattern
rag = RAGPattern(
    vector_store=crew.vector_store,
    collection=COLLECTION,
    top_k=5,
    config=config,
)

print("RAG Pattern initialized")
print(f"Collection: {COLLECTION}")


In [ ]:
# Ask a question about the codebase
question = "How does the CrewAI adapter integrate with MLFlow?"

print(f"Question: {question}")
print("="*50)

answer = rag.query(
    question=question,
    experiment_name="rag-queries",
)

print(f"\nAnswer:")
print(answer.output)


## 7. Embedding Provider Examples

The framework supports multiple embedding providers.


In [ ]:
# Test different embedding providers

# Ollama embeddings (default)
ollama_provider = EmbeddingProvider.create(
    provider="ollama",
    model="nomic-embed-text",
)

text = "This is a test sentence for embedding."
result = ollama_provider.embed(text)

print(f"Ollama Embedding:")
print(f"  Model: {result.model}")
print(f"  Dimension: {result.dimension}")
print(f"  Duration: {result.duration_ms:.2f}ms")
print(f"  Vector: [{result.embedding[0]:.4f}, {result.embedding[1]:.4f}, ...]")


## 8. Collection Statistics


In [ ]:
# Get collection statistics
stats = crew.get_stats(COLLECTION)

print("Collection Statistics")
print("="*50)
for key, value in stats.items():
    print(f"{key}: {value}")


## 9. Cleanup (Optional)

Clear the indexed collection if needed.


In [ ]:
# Uncomment to clear the collection
# crew.clear(COLLECTION)
# print(f"Collection '{COLLECTION}' cleared")


## Summary

This notebook demonstrated:

1. **Multi-Agent Crew** - Using specialized agents for code analysis
2. **Vector Storage** - Indexing code to LanceDB/ChromaDB
3. **Semantic Search** - Finding relevant code by meaning
4. **RAG Pattern** - Asking questions about the codebase
5. **Experiment Tracking** - MLFlow integration for observability

### Next Steps

- Try indexing your own repositories
- Experiment with different embedding models
- Use the RAG pattern for code Q&A
- View experiment data in MLFlow UI
- Explore traces in Jaeger (if configured)
